[Back to Computer Networks guideline](Computer-Networks.html)

## **IP Addressing, Forwarding, and Routers**

Chapter 2 ended when the default gateway received a local frame addressed to its link-layer interface. The gateway now removes that frame and exposes an IPv4 or IPv6 packet. From this point onward, the important questions are no longer "which MAC address is on this LAN?" but:

1. Is the packet structurally valid, and is it still allowed to travel?
2. Which destination prefix contains its address?
3. Which output interface and next hop should be used?
4. Can the packet fit on that output link?
5. If forwarding fails, what control message should be returned?

This chapter studies those decisions as the **Internet data plane**. Its running example is an IPv4 packet from `192.0.2.10` to a remote server at `198.51.100.8`. The packet crosses routers with different links and MTUs, while each router performs a bounded set of operations fast enough to handle every arriving packet.

::: {.callout-note}
On a first reading, prioritize prefixes and masks, longest-prefix matching, TTL, ICMP, and MTU. Router fabrics, TCAM, IPv4 fragmentation details, NAT variants, and IPv6 transition mechanisms explain implementation and operational trade-offs after the basic forwarding path is clear.
:::

The central separation is simple but essential: **routing creates or selects forwarding state; forwarding applies that state to individual packets**. Chapter 4 explains how routing protocols learn paths. Here, the forwarding table is an input whose behavior we inspect carefully.

### **Network-Layer Service and the Data Plane**

#### **Best-Effort Datagram Delivery**

IP offers a **connectionless datagram service**. Each packet carries a destination address and can be processed independently. The network does not establish a per-application circuit before forwarding ordinary IP traffic, and different packets from one flow may encounter different queues or even different paths.

The service is described as **best effort** because the base IP layer does not promise delivery, ordering, duplicate suppression, a minimum rate, or a maximum delay. A router may discard a packet because a queue is full, a header is invalid, the hop limit expires, a policy denies it, or no usable route exists. Best effort does not mean careless: routers perform precise forwarding, operators engineer capacity, link layers detect local corruption, and transport protocols can add reliability. It means those stronger guarantees are not part of the universal IP contract.

This deliberately small contract lets IP run over Ethernet, Wi-Fi, fiber systems, cellular tunnels, satellite links, and future technologies. Each link only needs a way to carry an IP packet to the next IP node; it need not reproduce every property of every other link.

#### **Forwarding vs Routing**

**Forwarding** is the local, per-packet operation that maps header fields and ingress context to an action such as send through interface `if3`, discard, mirror, or deliver locally. It belongs primarily to the data plane and must commonly run at line rate.

**Routing** is the wider process that discovers reachability, compares paths, reacts to topology changes, and installs selected routes. Static configuration and protocols such as OSPF or BGP can supply this state. Routing operates on a slower time scale than packet forwarding.

| Question | Forwarding data plane | Routing control plane |
|---|---|---|
| Main input | One packet and installed tables | Topology, reachability, policy, metrics |
| Main output | Packet action and output adjacency | Routes to install in forwarding state |
| Typical time scale | Nanoseconds to microseconds per packet | Milliseconds to minutes per event |
| Failure example | Queue drop or expired TTL | Stale, missing, looping, or withdrawn route |

A routing table shown by an operating system may contain protocol, preference, metric, and next-hop information. Hardware often receives a distilled **Forwarding Information Base (FIB)** optimized for lookup. The Routing Information Base (RIB), FIB, neighbor table, and link-layer forwarding database are related but distinct pieces of state.

#### **Per-Packet Processing at Hosts and Routers**

A host and a router both use IP forwarding logic, although a normal host primarily originates and terminates packets. A conceptual host decision is:

```text
HOST_SEND(packet)
    select a route using the destination and local policy
    choose a source address compatible with that route
    if the next hop is on a directly attached link
        resolve its link-layer address
    encapsulate the IP packet for the selected interface
    transmit or report a local error
```

A router's fast path is broader:

```text
ROUTER_FORWARD(frame, ingress)
    validate the received link frame and remove its link header
    validate the IP header and determine whether the packet is local
    decrement TTL or hop limit; reject the packet if it expires
    classify policy and find the longest matching destination prefix
    resolve the selected next-hop adjacency
    enforce output MTU and other actions
    queue, schedule, re-encapsulate, and transmit on the output link
```

The incoming and outgoing frames are different. Their source and destination link addresses describe adjacent interfaces on different links. The IP source and destination usually remain end to end, although TTL or hop limit changes at every router and a middlebox such as NAT may intentionally rewrite addresses.

**Checkpoint.** A switch in Chapter 2 used an exact destination MAC lookup within a LAN. An IP router uses hierarchical prefixes, can choose a next hop that is not the final destination, and prevents indefinite circulation through TTL or hop limit.

### **IPv4 Datagrams**

#### **IPv4 Header Fields**

An IPv4 datagram contains a variable-length header followed by payload. The ordinary header is 20 bytes; options can extend it to 60 bytes. Fields are transmitted in network byte order, meaning the most significant byte of a multi-byte integer appears first.

![IPv4 header fields and bit widths.](assets/ipv4-header.svg){fig-alt="IPv4 header layout with version, length, service, fragmentation, TTL, protocol, checksum, addresses, and options" width="88%"}

*Figure source: [Bruno Wenk, IPv4 Header, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:IPv4_Header.svg), licensed under CC BY-SA 3.0.*

| Field | Purpose | Important interpretation |
|---|---|---|
| Version | Identifies IPv4 | Value is `4` |
| Internet Header Length | Header size in 32-bit words | Minimum `5`, which means 20 bytes |
| DSCP and ECN | Traffic treatment and congestion signals | Policy information, not a bandwidth guarantee |
| Total Length | Header plus payload in bytes | Maximum representable value is 65,535 |
| Identification, Flags, Fragment Offset | Support IPv4 fragmentation | All fragments of one original datagram share an identifier |
| TTL | Limits forwarding lifetime | Routers reduce it; zero causes discard |
| Protocol | Identifies the next-layer payload | Examples: ICMP `1`, TCP `6`, UDP `17` |
| Header Checksum | Detects inconsistency in the IPv4 header | Does not cover payload |
| Source and Destination | 32-bit IPv4 addresses | Destination drives ordinary prefix lookup |
| Options and Padding | Optional control information and alignment | Variable parsing cost makes options uncommon in fast paths |

Total Length bounds one IPv4 packet, not an application message or transport stream. A large TCP transfer becomes many IP packets. Conversely, a small application write can still gain headers at several layers.

#### **Time to Live and Header Checksum**

TTL was historically time-oriented, but routers operationally treat it as a hop limit. [RFC 1812](https://datatracker.ietf.org/doc/html/rfc1812) requires a router to reduce TTL by at least one when forwarding. If the result is zero, the router discards the packet and normally sends ICMP Time Exceeded. This bounds the lifetime of a packet caught in a forwarding loop.

Because TTL changes, the IPv4 header checksum must also change at every hop. The checksum uses one's-complement addition over 16-bit header words. The sender calculates it with the checksum field zero. A receiver including the transmitted field should obtain all one bits before complement, commonly represented by a verification result of zero in software.

The checksum detects many accidental header errors, but it is not cryptographic integrity. An attacker can modify fields and recompute it. Ethernet CRC may protect a frame on one link, while TCP, UDP, or an application checksum may cover payload end to end; these checks have different scopes.

#### **Protocol Demultiplexing**

After a host accepts a packet addressed to itself, the Protocol field selects the next handler. TCP and UDP then use port numbers to identify transport endpoints. ICMP does not use transport ports; its Type and Code fields identify control messages. Encapsulated protocols can also appear directly inside IP.

Demultiplexing is not the same as forwarding. A transit router normally reads enough of the packet to apply its forwarding and policy rules but does not deliver the payload to a local TCP socket. If the destination is one of the router's own addresses, local input processing takes over.

In [1]:
import ipaddress
import struct


def internet_checksum(data: bytes) -> int:
    """Return the 16-bit one's-complement Internet checksum."""

    # The checksum works on 16-bit words. Pad an odd byte only for calculation.
    if len(data) % 2:
        data += b"\x00"

    total = sum(struct.unpack(f"!{len(data) // 2}H", data))
    # Fold every carry from bit 16 back into the low 16 bits.
    while total >> 16:
        total = (total & 0xFFFF) + (total >> 16)
    return (~total) & 0xFFFF


payload = b"hello router"
version_ihl = (4 << 4) | 5                    # IPv4, five 32-bit words
total_length = 20 + len(payload)
identification = 0x1234
flags_fragment = 0x4000                      # DF=1, fragment offset=0
ttl, protocol = 64, 17                       # UDP is protocol 17
source = ipaddress.IPv4Address("192.0.2.10").packed
destination = ipaddress.IPv4Address("198.51.100.8").packed

# First pack the header with a zero checksum, then calculate and insert it.
header_without_checksum = struct.pack(
    "!BBHHHBBH4s4s",
    version_ihl, 0, total_length, identification, flags_fragment,
    ttl, protocol, 0, source, destination,
)
checksum = internet_checksum(header_without_checksum)
header = struct.pack(
    "!BBHHHBBH4s4s",
    version_ihl, 0, total_length, identification, flags_fragment,
    ttl, protocol, checksum, source, destination,
)

# A correctly formed header verifies to zero when its checksum is included.
print(f"header bytes: {len(header)}")
print(f"total length: {total_length}")
print(f"TTL/protocol: {ttl}/{protocol}")
print(f"checksum:     0x{checksum:04x}")
print(f"verification: 0x{internet_checksum(header):04x}")

header bytes: 20
total length: 32
TTL/protocol: 64/17
checksum:     0x3c53
verification: 0x0000


### **IPv4 Addressing and Subnetting**

#### **Network and Host Portions**

An IPv4 address is a 32-bit value assigned to an interface or logical endpoint. Dotted decimal, such as `192.0.2.130`, is only a human-readable rendering of four octets. An address alone does not reveal which neighboring addresses are considered on-link. That interpretation requires a **prefix length**.

The interface notation `192.0.2.130/26` says that its first 26 bits identify the attached prefix and the remaining 6 bits identify positions within that prefix:

```text
address:  11000000.00000000.00000010.10|000010   192.0.2.130
mask:     11111111.11111111.11111111.11|000000   255.255.255.192
network:  11000000.00000000.00000010.10|000000   192.0.2.128
```

Applying bitwise AND between the address and mask clears the host-position bits:

$$
network = address \land mask_p.
$$

For this traditional subnet, the range is `192.0.2.128` through `192.0.2.191`. The all-zero host position names the subnet and the all-one position is its directed broadcast address, leaving `.129` through `.190` for conventional host assignment. The often-taught formula $2^{32-p}-2$ is therefore valid for many ordinary IPv4 subnets, but it is not universal: `/31` point-to-point links and `/32` host routes have explicit meanings and should not be forced into that rule.

#### **Subnet Masks and Prefix Lengths**

A valid ordinary mask has contiguous one bits followed by contiguous zero bits. Prefix notation is less error-prone than repeatedly converting masks:

| Prefix | Mask | Addresses in block | Common interpretation |
|---|---|---:|---|
| `/8` | `255.0.0.0` | $2^{24}$ | Very large aggregate |
| `/16` | `255.255.0.0` | $2^{16}$ | Medium aggregate |
| `/24` | `255.255.255.0` | 256 | Familiar small IPv4 subnet |
| `/26` | `255.255.255.192` | 64 | Four equal subdivisions of a `/24` |
| `/30` | `255.255.255.252` | 4 | Legacy point-to-point allocation |
| `/32` | `255.255.255.255` | 1 | One host address or exact route |

Subnetting borrows bits from the host portion to create smaller administrative or failure domains. Starting with `10.20.0.0/16`, using four additional subnet bits yields `/20` blocks. Each block advances by 16 in the third octet: `10.20.0.0/20`, `10.20.16.0/20`, and so on. The engineering choice balances address efficiency, broadcast-domain size, route aggregation, growth, and operational clarity.

#### **Classless Inter-Domain Routing**

Early classful addressing assumed fixed `/8`, `/16`, or `/24` boundaries according to leading address bits. That wasted addresses and made global routing growth difficult. **Classless Inter-Domain Routing (CIDR)**, described in [RFC 4632](https://datatracker.ietf.org/doc/html/rfc4632), lets a prefix end at any bit position.

CIDR supports two complementary operations:

- **Subnetting** divides a larger prefix into longer, smaller prefixes.
- **Aggregation** summarizes adjacent aligned prefixes with one shorter prefix.

For example, `198.51.100.0/25` and `198.51.100.128/25` cover exactly `198.51.100.0/24`, so an upstream router may advertise the `/24`. A more-specific route can still create an exception. Aggregation scales because distant routers can store one regional or provider prefix instead of every internal subnet.

Aggregation is only correct when the summarizing router can actually deliver all covered destinations or safely reject unavailable portions. An overly broad summary can attract traffic into a black hole. Prefixes must also be properly aligned: arbitrary adjacent-looking ranges are not necessarily representable by one CIDR block.

#### **Address Allocation and Special Addresses**

Global IPv4 space is allocated hierarchically through registries, providers, and organizations, allowing addresses and routes to reflect topology. Several blocks have special purposes and should not be interpreted as ordinary public destinations:

| Block | Meaning | Routability expectation |
|---|---|---|
| `10.0.0.0/8`, `172.16.0.0/12`, `192.168.0.0/16` | Private-use space from [RFC 1918](https://datatracker.ietf.org/doc/html/rfc1918) | Not globally routed on the public Internet |
| `127.0.0.0/8` | Loopback | Remains within one host |
| `169.254.0.0/16` | IPv4 link-local | Local link only |
| `224.0.0.0/4` | Multicast | Group delivery under multicast rules |
| `255.255.255.255/32` | Limited broadcast | Local link; routers do not forward it |
| `192.0.2.0/24`, `198.51.100.0/24`, `203.0.113.0/24` | Documentation examples | Reserved for examples, including this blog |

A private address is not automatically trusted, secret, or authenticated. It only belongs to a non-public allocation realm. Firewalls, identity, encryption, and application authorization solve different security problems.

In [2]:
from ipaddress import ip_interface, ip_network, collapse_addresses


interface = ip_interface("192.0.2.130/26")
network = interface.network

print("interface: ", interface)
print("mask:      ", network.netmask)
print("network:   ", network.network_address)
print("broadcast: ", network.broadcast_address)
print("block size:", network.num_addresses)
print("traditional usable range:",
      network.network_address + 1, "to", network.broadcast_address - 1)

# Adjacent, aligned prefixes can be represented by a shorter aggregate.
parts = [
    ip_network("198.51.100.0/25"),
    ip_network("198.51.100.128/25"),
    ip_network("203.0.113.0/24"),
]
aggregated = list(collapse_addresses(parts))
print("aggregated:", ", ".join(map(str, aggregated)))

interface:  192.0.2.130/26
mask:       255.255.255.192
network:    192.0.2.128
broadcast:  192.0.2.191
block size: 64
traditional usable range: 192.0.2.129 to 192.0.2.190
aggregated: 198.51.100.0/24, 203.0.113.0/24


### **Forwarding and Longest-Prefix Matching**

#### **Forwarding Tables and Next Hops**

A forwarding entry associates a destination prefix with an action. The action often names an outgoing interface and a **next-hop** address, but it can also indicate local delivery, discard, tunnelling, or a special policy path.

```text
prefix             next hop       interface
0.0.0.0/0          203.0.113.1    if1
10.0.0.0/8         10.0.0.2       if2
10.1.0.0/16        10.1.0.2       if3
10.1.2.0/24        on-link         if4
10.1.2.128/25      on-link         if5
```

An **on-link** or directly connected entry means the destination itself is the IP next hop on that interface. Otherwise, the router sends the packet to another router's address. The next hop must ultimately resolve to an adjacency that the output link can reach. Implementations may perform a recursive lookup when one route's next hop is reached through another route, then cache the resolved forwarding information.

The FIB does not store a complete list of every destination host. Hierarchical prefixes let one entry represent an entire range. The neighbor table then maps the selected local next-hop IP to link-layer delivery information.

#### **Longest-Prefix Match Algorithm**

Several prefixes can contain the same destination. **Longest-prefix matching (LPM)** selects the matching entry with the greatest prefix length, because it describes the smallest and therefore most specific address range.

![A longest-prefix matching table in which several routes match 10.1.2.130 and the slash 25 route wins.](assets/lpm-routing-table.svg){fig-alt="Destination 10.1.2.130 matches default, slash 8, slash 16, slash 24, and slash 25 routes; slash 25 wins" width="96%"}

The conceptual algorithm is:

```text
LONGEST_PREFIX_MATCH(destination, table)
    best <- no match
    for each entry in table
        if destination belongs to entry.prefix
            if best is absent or entry.prefix_length > best.prefix_length
                best <- entry
    return best
```

Metrics or administrative preferences choose among candidate routes during control-plane selection, and may break ties among equal-length installed alternatives. A shorter route with a lower metric does not defeat an available more-specific route during ordinary LPM. This distinction prevents the common mistake of interpreting forwarding as "choose the numerically cheapest row in the table."

The `/0` default route fixes zero leading bits, so every IPv4 destination matches it. It is selected only when no longer installed prefix matches. A host's familiar default gateway is therefore one instance of the same LPM rule, not an unrelated exception.

#### **Tries, TCAM, and Fast Lookup**

Scanning every row gives $O(n)$ lookup and is useful for explanation, but high-speed routers need more efficient structures:

- A **binary trie** follows one destination bit per level and remembers the deepest route encountered.
- Patricia and multi-bit tries compress paths or inspect several bits at once, trading memory layout against lookup steps.
- **TCAM** compares a key against many ternary patterns in parallel, where each stored bit can be `0`, `1`, or "do not care." Priority order returns the most specific programmed match.
- Software routers can combine prefix tries, caches, vectorized processing, and CPU-friendly memory layouts.

TCAM offers predictable high-speed lookup but consumes expensive silicon and power. Trie performance depends on memory accesses and update behavior. Real forwarding pipelines also classify ACLs, QoS, tunnel labels, and metadata; destination LPM is central but not the only lookup.

#### **Default Routes and Policy Actions**

Policy can influence forwarding before or after LPM. Examples include discarding spoofed sources, sending selected traffic through a firewall, choosing a table by virtual routing instance, or applying equal-cost multipath hashing. These actions should be made explicit because they can explain why an observed packet does not follow the route a simple destination-only lookup predicts.

Chapter 4 asks how the prefixes reached the table and whether they represent stable, loop-free paths. This chapter assumes the entries are already installed and asks exactly how one packet uses them.

In [3]:
from dataclasses import dataclass
from ipaddress import IPv4Address, IPv4Network


@dataclass(frozen=True)
class Route:
    prefix: IPv4Network
    next_hop: str
    interface: str


routes = [
    Route(IPv4Network("0.0.0.0/0"), "203.0.113.1", "if1"),
    Route(IPv4Network("10.0.0.0/8"), "10.0.0.2", "if2"),
    Route(IPv4Network("10.1.0.0/16"), "10.1.0.2", "if3"),
    Route(IPv4Network("10.1.2.0/24"), "on-link", "if4"),
    Route(IPv4Network("10.1.2.128/25"), "on-link", "if5"),
]


def longest_prefix_match(destination: str, table: list[Route]) -> Route | None:
    """Return the most-specific route containing the destination address."""

    address = IPv4Address(destination)
    matches = [route for route in table if address in route.prefix]
    return max(matches, key=lambda route: route.prefix.prefixlen, default=None)


for destination in ["10.1.2.130", "10.1.2.42", "203.0.113.9"]:
    route = longest_prefix_match(destination, routes)
    print(
        f"{destination:15} -> {str(route.prefix):18} "
        f"via {route.next_hop:11} {route.interface}"
    )

10.1.2.130      -> 10.1.2.128/25      via on-link     if5
10.1.2.42       -> 10.1.2.0/24        via on-link     if4
203.0.113.9     -> 0.0.0.0/0          via 203.0.113.1 if1


### **Router Architecture**

#### **Input Ports and Packet Classification**

A router is not merely a CPU with several cables. High-rate systems divide packet work into a data path and a control processor. The data path is implemented with NIC logic, switching ASICs, network processors, kernel fast paths, or combinations of them.

![Router data plane from input parsing and longest-prefix lookup through the switching fabric to output queues and scheduling.](assets/router-data-plane.svg){fig-alt="Router input port, validation and longest-prefix lookup, switch fabric, output queues, scheduler, and control processor" width="98%"}

At an input port, physical and link logic receives a frame and verifies link-level conditions. IP parsing then checks that enough bytes exist, the version and header length are plausible, the IPv4 checksum is valid, and the packet is not obviously malformed. Classification can use destination, source, protocol, ports, ingress interface, tunnel context, and policy metadata.

The lookup result is more than an interface number. It can include the next-hop adjacency, output rewrite template, queue class, counters, and actions such as drop, mirror, encapsulate, or send to the control processor. Preparing this metadata once lets later stages avoid repeating expensive interpretation.

#### **Switching Fabrics**

The **switching fabric** moves packets from input-side storage to the selected output side. Common conceptual designs include:

- **Switching through memory:** an input writes packet data to shared memory and an output reads it. Memory bandwidth can become the limiting shared resource.
- **Shared bus:** inputs place packets on a common internal bus observed by outputs. Arbitration prevents simultaneous conflicting use.
- **Crossbar or interconnection network:** several input-output transfers can occur concurrently when they do not contend for the same resources.

If $N$ input ports can each receive at rate $R$, an ideal fabric that never bottlenecks would need aggregate capacity on the order of $NR$, often with additional speedup because packets arrive in bursts and internal transfers have overhead. Raw fabric bandwidth is not enough by itself; arbitration must decide which input may reach an output when requests conflict.

#### **Output Ports, Queues, and Scheduling**

Even a very fast fabric cannot make a 10 Gb/s output transmit 20 Gb/s of simultaneous arrivals. The excess waits in a queue. Once finite buffering fills, the router must drop or mark packets.

A scheduler selects the next packet:

| Scheduler idea | Decision | Strength and cost |
|---|---|---|
| FIFO | Oldest queued packet first | Simple, but no service differentiation |
| Strict priority | Serve highest non-empty class | Low delay for priority traffic; can starve lower classes |
| Round robin | Rotate among active queues | Simple sharing, insensitive to packet size unless refined |
| Weighted fair scheduling | Approximate configured service shares | Better isolation, more state and computation |

Queueing introduces variable delay; excessive unmanaged buffering can keep links busy while producing long latency, known as bufferbloat. Congestion control and active queue management are developed in Chapter 6. Here, the key fact is causal: queues appear whenever short-term arrival demand exceeds the service rate of a fabric, output link, or processing stage.

#### **Head-of-Line Blocking and Buffer Placement**

With one FIFO at each input, the first packet may be waiting for a busy output while packets behind it need idle outputs. Those later packets cannot pass the front packet, creating **head-of-line (HOL) blocking**. Virtual output queues separate packets by desired output and reduce this problem, but require more queues and scheduling state.

Buffers can be placed at inputs, outputs, or shared internally. No placement removes overload: if offered traffic persistently exceeds output capacity, finite storage eventually fills. Architecture determines how efficiently temporary bursts are absorbed, how memory bandwidth scales, and where fairness decisions occur.

#### **Control Processor and Fast Path**

The control processor runs routing protocols, management services, monitoring, and exception handling, then programs data-plane tables. Ordinary transit packets should not require a general CPU interrupt. Packets needing uncommon handling, such as some options, unresolved adjacencies, or control-plane destinations, may be **punted** to a slow path with strict rate limits.

This separation explains an operational pattern: a router can forward already-installed traffic at high speed while its management interface is busy, or conversely remain reachable for management while one hardware forwarding component fails. "The router CPU is low" does not prove that links, queues, fabric, or FIB resources are healthy.

### **Maximum Transmission Unit and Fragmentation**

#### **Link MTU, Path MTU, and Transport MSS**

The **Maximum Transmission Unit (MTU)** is the largest network-layer packet a link can carry in one link-layer unit under the relevant configuration. Ethernet commonly exposes an IP MTU of 1500 bytes, but other values are valid. The **Path MTU (PMTU)** is the minimum usable MTU across all links on a path:

$$
PMTU = \min(MTU_1, MTU_2, \ldots, MTU_k).
$$

MTU includes the IP header. TCP's **Maximum Segment Size (MSS)** describes TCP payload per segment, so a typical IPv4/TCP calculation without options is `1500 - 20 - 20 = 1460` bytes. MTU and MSS are related but not interchangeable.

#### **IPv4 Fragmentation and Reassembly**

If an IPv4 datagram exceeds an output MTU and the Don't Fragment flag is clear, a router can split it. Every fragment receives its own IPv4 header. Non-final fragment payload lengths are multiples of eight bytes because Fragment Offset is measured in eight-byte units.

![A 4000-byte IPv4 datagram split into three fragments for a 1500-byte output MTU, plus the Path MTU Discovery alternative.](assets/mtu-fragmentation-pmtud.svg){fig-alt="IPv4 fragmentation into 1500, 1500, and 1040 byte packets with offsets zero, 185, and 370, and DF path MTU discovery" width="98%"}

For the figure's 4000-byte datagram with a 20-byte header and a 1500-byte MTU:

$$
max\_aligned\_payload = \left\lfloor\frac{1500-20}{8}\right\rfloor 8 = 1480.
$$

The first two fragments carry 1480 payload bytes and set More Fragments (`MF=1`). Their offsets are `0` and `1480/8 = 185`. The final payload is 1020 bytes, begins at original byte 2960, uses offset `2960/8 = 370`, and clears `MF`.

Reassembly occurs only at the final destination, keyed by source, destination, protocol, and Identification along with fragment ranges. If one fragment is lost, the complete original datagram cannot be delivered. Fragmentation also multiplies headers, complicates filtering and load balancing, and can amplify the effect of one loss. [RFC 8900](https://datatracker.ietf.org/doc/html/rfc8900) documents why IP fragmentation is fragile in modern networks.

#### **Path MTU Discovery**

With IPv4 Don't Fragment (`DF=1`), a router unable to forward an oversized packet drops it and should send ICMP Destination Unreachable with the fragmentation-needed code and usable MTU. The source reduces packet size. IPv6 uses an ICMPv6 Packet Too Big message because IPv6 routers never fragment transit packets.

Classic PMTUD can fail when a firewall discards the necessary ICMP message, producing a **PMTU black hole**: small packets work while larger ones stall. Packetization Layer PMTU Discovery in [RFC 8899](https://datatracker.ietf.org/doc/html/rfc8899) confirms usable size with transport-layer probes and does not rely solely on receiving ICMP.

#### **Encapsulation and Effective MTU**

A tunnel adds an outer IP header and possibly UDP, security, or tunnel headers. If the underlay MTU is 1500 and encapsulation adds 50 bytes, the safe inner packet may be only 1450 bytes:

$$
effective\ inner\ MTU = underlay\ MTU - encapsulation\ overhead.
$$

Ignoring overhead causes fragmentation, drops with DF set, or repeated retransmission. Operators address this with larger underlay MTUs, smaller tunnel-interface MTUs, correct ICMP handling, transport probing, or TCP MSS adjustment. MSS adjustment only influences TCP; it does not solve oversized UDP or other IP traffic.

In [4]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Fragment:
    total_length: int
    payload_length: int
    offset_units: int
    more_fragments: bool


def fragment_ipv4(total_length: int, mtu: int, header_length: int = 20) -> list[Fragment]:
    """Calculate IPv4 fragments; options and copied-option rules are omitted."""

    if header_length < 20 or header_length % 4:
        raise ValueError("IPv4 header length must be a multiple of four")
    if total_length <= mtu:
        return [Fragment(total_length, total_length - header_length, 0, False)]

    # Every non-final fragment payload must fit and be divisible by eight.
    aligned_payload = ((mtu - header_length) // 8) * 8
    if aligned_payload <= 0:
        raise ValueError("MTU is too small for this IPv4 header")

    remaining = total_length - header_length
    offset_bytes = 0
    fragments = []
    while remaining:
        payload = min(aligned_payload, remaining)
        remaining -= payload
        fragments.append(
            Fragment(
                total_length=header_length + payload,
                payload_length=payload,
                offset_units=offset_bytes // 8,
                more_fragments=remaining > 0,
            )
        )
        offset_bytes += payload
    return fragments


for number, fragment in enumerate(fragment_ipv4(4000, 1500), start=1):
    print(
        f"fragment {number}: total={fragment.total_length:4}, "
        f"payload={fragment.payload_length:4}, offset={fragment.offset_units:3}, "
        f"MF={int(fragment.more_fragments)}"
    )

fragment 1: total=1500, payload=1480, offset=  0, MF=1
fragment 2: total=1500, payload=1480, offset=185, MF=1
fragment 3: total=1040, payload=1020, offset=370, MF=0


### **Internet Control Message Protocol**

#### **Errors, Diagnostics, and Reachability**

The **Internet Control Message Protocol (ICMP)** carries network-layer error reports and operational information. ICMPv4 is specified by [RFC 792](https://datatracker.ietf.org/doc/html/rfc792) and later updates; ICMPv6 is a distinct but more integral part of IPv6.

Important messages include:

| Situation | Typical control message | What it proves |
|---|---|---|
| TTL reaches zero | Time Exceeded | One router processed and discarded this packet |
| No usable route or prohibited delivery | Destination Unreachable with a code | A node could not complete a particular delivery step |
| IPv4 DF packet is too large | Destination Unreachable: fragmentation needed | A smaller packet is required at that point |
| IPv6 packet is too large | ICMPv6 Packet Too Big | Source must reduce IPv6 packet size |
| Diagnostic query reaches a responding host | Echo Reply | ICMP request and reply paths worked at that time |

An ICMP error includes part of the offending packet so the sender can associate it with a flow or socket. ICMP is not transport reliability: it does not acknowledge every packet, and error messages themselves can be filtered, rate-limited, lost, or suppressed to avoid loops and storms. Absence of ICMP therefore does not prove absence of a failure.

#### **Echo Request and Echo Reply**

`ping` sends Echo Requests and measures Echo Replies. It can estimate round-trip time, loss among probes, and reachability for that ICMP exchange. It cannot by itself prove that DNS, TCP port 443, application authentication, or high-rate data transfer works. A firewall may allow the application and deny echo, or the reverse.

Interpret latency as a round trip through two potentially different paths plus processing and queueing. One slow reply may reflect temporary queueing; repeated distributions are more informative than a single value.

#### **Time Exceeded and Traceroute**

Traceroute sends probes with increasing TTL or IPv6 Hop Limit. The first probe expires at the first router, the second at the second router, and so on. ICMP Time Exceeded messages reveal the source addresses of responding interfaces.

![Traceroute probes with TTL values one through four and the ICMP Time Exceeded replies that identify successive routers.](assets/icmp-traceroute.svg){fig-alt="Traceroute sends probes with increasing TTL and receives ICMP Time Exceeded from each router before reaching the destination" width="98%"}

Implementations use UDP, ICMP Echo, or TCP probes. The destination response depends on the method: a UDP traceroute may receive Port Unreachable, an echo-based trace may receive Echo Reply, and a TCP trace may receive a TCP response. Consequently, different traceroute modes can cross different policy or load-balancing treatment.

An asterisk means a reply was not observed before timeout. The router may still have forwarded the probe but suppressed or rate-limited ICMP. Interface addresses in replies identify response sources, not necessarily complete router identities. Forward and return paths can differ, and per-flow load balancing can show different hops for probes whose header fields change.

ICMP provides evidence about one attempted path. Combine it with route tables, packet capture, application tests, and repeated measurements before concluding exactly where a fault lies.

### **Host Configuration and Address Translation**

#### **Dynamic Host Configuration Protocol**

A new host may know neither its usable address nor the local router and DNS resolver. **DHCP** supplies leased configuration over UDP. The familiar DORA sequence is:

1. **Discover:** the client broadcasts to locate available DHCP servers.
2. **Offer:** a server proposes an address and options.
3. **Request:** the client identifies the offer it accepts or requests renewal.
4. **Acknowledge:** the server confirms the lease and parameters.

![Typical DHCP discovery, offer, request, and acknowledgment exchange.](assets/dhcp-session.svg){fig-alt="Client and DHCP server exchange Discover, Offer, Request, and Acknowledge messages" width="72%"}

*Figure source: [DHCP session diagram, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:DHCP_session_en-2.svg), reusable under the license shown on the source page.*

Initial messages often use client port 68, server port 67, broadcasts, transaction identifiers, and the special source `0.0.0.0` because the client is not configured yet. In a routed enterprise, a **DHCP relay** receives the local broadcast and forwards enough context to a remote server, avoiding one server per VLAN.

The address is only one option. DHCP can provide a prefix mask, default routers, DNS servers, domain search information, lease time, and other network-specific parameters. A lease is time-bounded state: the client renews before expiry, while the server tracks allocation. A successful Wi-Fi association without successful DHCP can therefore produce local link connectivity but no normal IP configuration.

DHCP does not prove that a client is trustworthy, and a received default gateway or DNS server can redirect traffic. Managed networks use switch controls, authentication, monitoring, and cryptographic application protocols to reduce the effect of unauthorized configuration.

#### **Network Address Translation and Port Translation**

**Network Address Translation (NAT)** rewrites address information between realms. The common home-router form is **Network Address and Port Translation (NAPT/PAT)**: many private endpoints share one public IPv4 address by receiving distinct translated source ports.

![Several private hosts share one public IPv4 address through a NAT mapping.](assets/nat-concept.svg){fig-alt="Private network clients pass through a NAT router that rewrites them to one public IPv4 address" width="78%"}

*Figure source: [Mro, NAT Concept, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:NAT_Concept-en.svg), licensed under CC BY-SA 4.0.*

Suppose `10.0.0.24:51514` sends UDP to `198.51.100.8:443`. A PAT device might create:

```text
inside local                 outside-facing translation
10.0.0.24:51514  <------>   203.0.113.9:40000
```

The outbound source becomes `203.0.113.9:40000`; the destination remains the server. A reply to public port 40000 is mapped back to the private tuple. The device must update affected IP and transport checksums and maintain protocol-specific timeout state. Other internal flows receive different public ports even when they share the same address.

NAT terminology and mapping behavior vary. Some mappings depend only on the internal endpoint; stricter forms also bind the remote address and port. Static port forwarding creates an explicit inbound mapping. Carrier-grade NAT adds another provider-operated translation layer, so a subscriber may not control the public address at all.

#### **Benefits, Costs, and End-to-End Consequences of NAT**

PAT conserves public IPv4 addresses and lets an organization renumber internally without changing every outside-visible flow identity. Its state also blocks unsolicited inbound packets that lack a mapping, but **NAT is not equivalent to a firewall**. A firewall expresses security policy; translation expresses address/port mapping. Either can exist without the other.

Translation weakens the original end-to-end model:

- An outside peer cannot initiate a connection without static mapping, traversal, or relay support.
- Protocols embedding addresses inside payload may need application-aware handling.
- Peer-to-peer and real-time applications use mechanisms such as STUN, TURN, ICE, or relays.
- Logs need address, port, protocol, and precise time to identify one subscriber flow.
- Stateful mappings must survive or be replicated across device failover.
- Port exhaustion can limit simultaneous translations even when bandwidth remains.

Encryption above transport, such as TLS, normally survives address translation because NAT need not read application plaintext. Protocols that authenticate IP headers directly or encrypt transport headers can require NAT-aware designs.

In [5]:
from dataclasses import dataclass


@dataclass(frozen=True)
class Flow:
    source_ip: str
    source_port: int
    destination_ip: str
    destination_port: int
    protocol: str = "UDP"


class PortAddressTranslator:
    """A small endpoint-dependent PAT model for outbound flows."""

    def __init__(self, public_ip: str, first_port: int = 40000):
        self.public_ip = public_ip
        self.next_port = first_port
        self.outbound_table = {}   # original five-tuple -> public port
        self.inbound_table = {}    # (protocol, public port, remote IP, port) -> client

    def translate_outbound(self, flow: Flow) -> Flow:
        key = (flow.source_ip, flow.source_port, flow.destination_ip,
               flow.destination_port, flow.protocol)
        if key not in self.outbound_table:
            public_port = self.next_port
            self.next_port += 1
            self.outbound_table[key] = public_port
            reverse_key = (flow.protocol, public_port,
                           flow.destination_ip, flow.destination_port)
            self.inbound_table[reverse_key] = (flow.source_ip, flow.source_port)

        public_port = self.outbound_table[key]
        return Flow(self.public_ip, public_port,
                    flow.destination_ip, flow.destination_port, flow.protocol)

    def translate_inbound(self, flow: Flow) -> Flow | None:
        reverse_key = (flow.protocol, flow.destination_port,
                       flow.source_ip, flow.source_port)
        private_endpoint = self.inbound_table.get(reverse_key)
        if private_endpoint is None:
            return None                  # No mapping: this toy NAT drops the packet.
        private_ip, private_port = private_endpoint
        return Flow(flow.source_ip, flow.source_port,
                    private_ip, private_port, flow.protocol)


nat = PortAddressTranslator("203.0.113.9")
request = Flow("10.0.0.24", 51514, "198.51.100.8", 443)
public_request = nat.translate_outbound(request)
reply = Flow("198.51.100.8", 443,
             public_request.source_ip, public_request.source_port)
private_reply = nat.translate_inbound(reply)

print("private request:", request)
print("public request: ", public_request)
print("translated reply:", private_reply)

private request: Flow(source_ip='10.0.0.24', source_port=51514, destination_ip='198.51.100.8', destination_port=443, protocol='UDP')
public request:  Flow(source_ip='203.0.113.9', source_port=40000, destination_ip='198.51.100.8', destination_port=443, protocol='UDP')
translated reply: Flow(source_ip='198.51.100.8', source_port=443, destination_ip='10.0.0.24', destination_port=51514, protocol='UDP')


### **IPv6**

#### **IPv6 Addressing and Header Design**

IPv6 expands addresses from 32 to 128 bits and simplifies the base header. An address is written as eight hexadecimal groups. Leading zeros within a group can be removed, and one longest run of all-zero groups can be compressed with `::`:

```text
2001:0db8:0000:0000:0000:0000:0000:0080
2001:db8::80
```

As in IPv4, a prefix length describes topology. `2001:db8:42:10::/64` identifies a 64-bit prefix; it does not imply that addresses should be scanned or that every value is active. Common scopes include global unicast, link-local `fe80::/10`, and multicast `ff00::/8`. IPv6 has no broadcast address; multicast and anycast cover relevant delivery patterns.

![IPv6 fixed base header fields and bit widths.](assets/ipv6-header.svg){fig-alt="IPv6 base header with version, traffic class, flow label, payload length, next header, hop limit, and 128-bit addresses" width="88%"}

*Figure source: [IPv6 header based on RFC 8200, Wikimedia Commons](https://commons.wikimedia.org/wiki/File:IPv6_header-en.svg), licensed under CC BY-SA 4.0.*

The fixed base header is 40 bytes:

| Field | Purpose |
|---|---|
| Version | Value `6` |
| Traffic Class | Differentiated treatment and congestion indication |
| Flow Label | Identifies packets belonging to a flow for suitable handling |
| Payload Length | Bytes after the 40-byte base header |
| Next Header | Extension-header type or upper-layer protocol |
| Hop Limit | IPv6 equivalent of IPv4 TTL |
| Source and Destination | 128-bit addresses |

IPv6 removes the base-header checksum, avoiding a checksum update solely because Hop Limit changes. Link and transport checks remain. It also moves optional information into extension headers, so the common base header has a stable layout.

#### **Neighbor Discovery and Stateless Configuration**

IPv6 Neighbor Discovery uses ICMPv6 for neighbor address resolution, router discovery, prefix advertisement, reachability, and duplicate-address detection. A host always has a link-local address for local IPv6 functions. Router Advertisements can provide a prefix and default-router information.

With **Stateless Address Autoconfiguration (SLAAC)**, a host forms an address from an advertised prefix and an interface identifier, then checks for duplication. Modern systems may use stable or temporary privacy-oriented interface identifiers rather than embedding a hardware address. DHCPv6 can provide managed addresses or additional options, but the default router is learned through Router Advertisements rather than a DHCPv6 router option.

#### **Extension Headers and Fragmentation**

Extension headers form a chain selected by Next Header values. Examples include Hop-by-Hop Options, Routing, Fragment, Destination Options, and IPsec-related headers. They provide extensibility but can complicate fast-path parsing and policy when a device must follow a long or unusual chain.

Under [RFC 8200](https://datatracker.ietf.org/doc/rfc8200/), IPv6 routers do **not** fragment packets. A source that needs fragmentation adds a Fragment header, normally after learning the path limitation from ICMPv6 Packet Too Big or transport probing. The IPv6 minimum link MTU and source-fragmentation model make correct ICMPv6 handling especially important.

#### **Dual Stack, Tunnelling, and Translation**

Deployment is incremental, so networks combine:

- **Dual stack:** endpoints and infrastructure run IPv4 and IPv6 concurrently.
- **Tunnelling:** one protocol is encapsulated across infrastructure carrying another.
- **Translation:** gateways map between IPv4 and IPv6 semantics, often with DNS assistance such as NAT64/DNS64.

Dual stack preserves native behavior but operates two protocol families. Tunnels cross unsupported regions but add overhead and MTU complexity. Translation helps IPv6-only clients reach IPv4 services but introduces state or semantic limitations. Application connection racing, often called Happy Eyeballs, reduces delay when one family is configured but unusable.

| Property | IPv4 | IPv6 |
|---|---|---|
| Address width | 32 bits | 128 bits |
| Base header | Variable, normally 20 bytes | Fixed 40 bytes |
| Header checksum | Yes | No |
| Transit-router fragmentation | Possible when DF is clear | Not permitted |
| Local resolution | ARP | ICMPv6 Neighbor Discovery |
| Broadcast | Supported | Replaced by multicast/other mechanisms |

IPv6 is not simply "IPv4 with more addresses." Address configuration, local control, fragmentation responsibility, extension parsing, and operational security need IPv6-specific treatment.

### **Forwarding Beyond Basic IP**

#### **Tunnels and Virtual Interfaces**

A **tunnel** treats one packet as payload inside another. The outer header carries the packet between tunnel endpoints; after decapsulation, the inner packet continues according to its own header. VPNs, overlays, mobility systems, and network virtualization use this pattern.

From the forwarding plane's perspective, a tunnel endpoint behaves like a virtual interface with encapsulation and decapsulation actions. The underlay routes the outer address, while the overlay interprets the inner address. Operators must identify which layer a route, MTU, packet capture, and failure belong to. A successful underlay ping does not prove that an overlay mapping or security association is correct.

#### **Multiprotocol Label Switching**

**MPLS** places one or more short labels between link and network headers. An ingress router classifies a packet into a forwarding equivalence class and pushes a label. Transit label-switching routers swap or pop labels according to a table; an egress router exposes the original payload.

Label lookup can support engineered paths, VPN separation, and compact forwarding state. MPLS is not an alternative to having a control plane: protocols or controllers still distribute labels and determine which paths they represent. It also does not remove IP; many MPLS networks carry IP packets and use IP reachability to build control-plane sessions.

#### **Middleboxes and Service Functions**

Modern paths commonly contain devices whose action depends on more than destination prefixes:

- firewalls allow or deny traffic using policy and connection state;
- load balancers select a service instance and may rewrite endpoints;
- NAT devices translate addresses and ports;
- intrusion detection or prevention inspects traffic patterns;
- service chains steer selected traffic through ordered functions.

These devices widen forwarding from `destination -> next hop` to `packet fields + state + policy -> action`. They can improve security and service operation, but hidden state complicates troubleshooting, asymmetric routing, failover, and end-to-end assumptions. Encryption protects content while also limiting which intermediaries can inspect it.

The durable mental model is a pipeline: parse, classify, look up state, apply actions, queue, and transmit. Basic IP LPM is one especially important instance of that general forwarding pipeline.

### **Building a Minimal Longest-Prefix-Match Router**

A teaching router can combine the chapter's decisions without pretending to implement a production device. The model below receives already-validated IPv4 metadata, performs TTL and LPM checks, applies output MTU behavior, and returns an explicit action.

```text
MINIMAL_FORWARD(packet)
    if packet.TTL <= 1
        discard and request ICMP Time Exceeded
    route <- LONGEST_PREFIX_MATCH(packet.destination)
    if no route exists
        discard and request ICMP Destination Unreachable
    decrement packet.TTL
    if packet.length > route.output_MTU
        if DF is set
            discard and request ICMP fragmentation needed
        else
            fragment for the output MTU
    resolve or use the route's next-hop adjacency
    queue packet on the selected output
```

The order matters. A packet whose TTL expires must not be forwarded merely because a route exists. A packet that matches a route can still fail the output-link constraint. ICMP generation is a separate action subject to its own source-address selection, route lookup, policy, and rate limit.

Production forwarding additionally handles IPv4 options, IPv6 extension headers, ACLs, ECN, QoS, multicast, equal-cost hashing, tunnel actions, counters, checksum updates, adjacency failure, and hardware resource limits. The small model is useful because each result still corresponds to a real category of router behavior.

In [6]:
from dataclasses import dataclass, replace


@dataclass(frozen=True)
class Packet:
    source: str
    destination: str
    ttl: int
    total_length: int
    dont_fragment: bool
    protocol: str = "UDP"


OUTPUT_MTU = {"if1": 1500, "if2": 9000, "if3": 1500, "if4": 1500, "if5": 1500}


def forward_ipv4(packet: Packet) -> dict:
    """Make a transparent forwarding decision for one simplified IPv4 packet."""

    if packet.ttl <= 1:
        return {"action": "ICMP Time Exceeded", "packet": None}

    route = longest_prefix_match(packet.destination, routes)
    if route is None:
        return {"action": "ICMP Destination Unreachable", "packet": None}

    forwarded = replace(packet, ttl=packet.ttl - 1)
    mtu = OUTPUT_MTU[route.interface]
    if forwarded.total_length > mtu:
        if forwarded.dont_fragment:
            return {
                "action": "ICMP fragmentation needed",
                "reported_mtu": mtu,
                "packet": None,
            }
        fragments = fragment_ipv4(forwarded.total_length, mtu)
        return {
            "action": "forward fragments",
            "interface": route.interface,
            "next_hop": route.next_hop,
            "ttl": forwarded.ttl,
            "fragment_lengths": [fragment.total_length for fragment in fragments],
        }

    return {
        "action": "forward",
        "interface": route.interface,
        "next_hop": route.next_hop,
        "ttl": forwarded.ttl,
        "length": forwarded.total_length,
    }


examples = [
    Packet("192.0.2.10", "10.1.2.130", 64, 1200, True),
    Packet("192.0.2.10", "198.51.100.8", 1, 1200, True),
    Packet("192.0.2.10", "198.51.100.8", 64, 2000, True),
    Packet("192.0.2.10", "198.51.100.8", 64, 4000, False),
]

for packet in examples:
    print(f"{packet.destination:15} ttl={packet.ttl:2} len={packet.total_length:4} "
          f"DF={int(packet.dont_fragment)} -> {forward_ipv4(packet)}")

10.1.2.130      ttl=64 len=1200 DF=1 -> {'action': 'forward', 'interface': 'if5', 'next_hop': 'on-link', 'ttl': 63, 'length': 1200}
198.51.100.8    ttl= 1 len=1200 DF=1 -> {'action': 'ICMP Time Exceeded', 'packet': None}
198.51.100.8    ttl=64 len=2000 DF=1 -> {'action': 'ICMP fragmentation needed', 'reported_mtu': 1500, 'packet': None}
198.51.100.8    ttl=64 len=4000 DF=0 -> {'action': 'forward fragments', 'interface': 'if1', 'next_hop': '203.0.113.1', 'ttl': 63, 'fragment_lengths': [1500, 1500, 1040]}


### **Observing and Troubleshooting the IP Data Plane**

Start with the smallest failed decision rather than assuming "the Internet is down."

| Evidence | Windows command | Question answered |
|---|---|---|
| Interface addresses and prefixes | `Get-NetIPAddress` or `ipconfig /all` | Which addresses and prefix lengths are configured? |
| Installed routes | `Get-NetRoute` or `route print` | Which prefix, next hop, and interface should win? |
| Neighbor state | `Get-NetNeighbor` or `arp -a` | Is the selected on-link next hop resolved? |
| Reachability and RTT | `ping <address>` | Does this ICMP exchange complete? |
| Hop-by-hop responses | `tracert <address>` | Where do increasing-TTL probes receive replies? |
| Specific TCP service | `Test-NetConnection <host> -Port 443` | Can a TCP connection reach this application port? |

Linux equivalents include `ip address`, `ip route get <address>`, `ip neighbor`, `ping`, `tracepath`, and `traceroute`. On macOS, `ifconfig`, `route -n get`, `arp -a`, `ping`, and `traceroute` expose similar evidence. Command output and privileges differ by platform.

Useful Wireshark display filters include:

```text
ip
ip.addr == 198.51.100.8
ip.ttl <= 2
ip.flags.df == 1
ip.fragment
icmp
ipv6
icmpv6
```

Follow a packet in order:

1. Confirm the host has the intended source address and prefix.
2. Ask the route table which entry wins for the exact destination.
3. Confirm the selected next hop is reachable on the named interface.
4. Capture the outgoing packet and check destination, TTL, DF, and length.
5. Look for ICMP errors, retransmissions, or missing replies.
6. Compare small and large probes when an MTU black hole is suspected.
7. If NAT is present, correlate both sides using address, port, protocol, and time.

| Symptom | Plausible data-plane causes | Discriminating evidence |
|---|---|---|
| Same-subnet traffic works, remote traffic fails | Missing/wrong default route, gateway failure, upstream policy | Route lookup and gateway reachability |
| One prefix fails while others work | More-specific bad route, policy action, remote withdrawal | Exact LPM result and traceroute comparison |
| Small requests work, larger transfers stall | PMTU black hole, tunnel overhead, blocked ICMP | Packet sizes, DF, ICMP, `tracepath` behavior |
| Outbound works, unsolicited inbound fails | Stateful firewall or missing NAT mapping | NAT/firewall state and public tuple |
| Traceroute shows stars but application works | ICMP filtering or rate limiting | Successful service connection despite no hop reply |
| IPv4 works, IPv6 fails | RA, NDP, IPv6 route, firewall, or PMTU issue | Separate IPv6 address/route/ICMPv6 checks |

Packet capture can expose private data and may require administrative permission. Capture only where authorized, minimize retained payload, and prefer controlled test addresses and traffic.

### **Comparison and Summary**

The IP data plane turns one destination address into one constrained forwarding action. Its concepts are easiest to retain as distinctions:

| Do not confuse | First concept | Second concept |
|---|---|---|
| Address vs prefix | Identifies an interface or endpoint | Identifies an address range used for topology and lookup |
| Subnet mask vs default route | Defines prefix bits for an attached network | Matches everything only when no more-specific route wins |
| Forwarding vs routing | Applies installed state to packets | Learns and selects the state |
| LPM vs shortest path | Chooses most-specific installed prefix | Helps a control plane choose a route to install |
| Next hop vs final destination | Adjacent IP node used for this forwarding step | End-to-end IP destination in the packet |
| MTU vs MSS | Maximum network-layer packet for a link/path | Maximum TCP payload per segment |
| ICMP failure vs silence | Explicit control evidence | Ambiguous: filtering, loss, or no generated response |
| NAT vs firewall | Rewrites address/port realms | Enforces traffic policy |
| IPv4 vs IPv6 fragmentation | Transit routers may fragment when allowed | Only the IPv6 source fragments |

For the running HTTPS packet, the gateway removes the first-hop frame, verifies IPv4, reduces TTL, performs LPM on `198.51.100.8`, checks the output MTU, possibly applies NAT or another policy action, waits in an output queue, and creates a new frame for the chosen next hop. Every subsequent router repeats a related data-plane pipeline.

The unresolved question is where the FIB entries came from and whether independent routers agree on a stable path. Chapter 4 moves to the control plane: distance-vector and link-state reasoning inside an autonomous system, BGP policy between autonomous systems, convergence, failures, and routing security.